# 使用 PROC FCMP 构建可复用的精算定价函数库

## 执行摘要

一家财产与意外险保险公司将其核心定价数学——纯保费、费用/风险加成、有限波动信度混合以及贴现准备金估计——封装为 **PROC FCMP** 中的自定义函数和一个多输出子程序。编译好的函数库通过 `CMPLIB=` 系统选项注册，随后在一个逐行处理合成的 100 份保单组合的 DATA 步中被调用。本笔记本中每一个定价结果——信度因子 `z`、混合纯保费、收取保费以及贴现后的案件准备金——都是由这些编译好的例程产生，而不是靠内联算术。结果：隐含赔付率落在**农村 60.5%**、**郊区 55.8%**、**城市 63.6%**——在每个细分市场都稳妥地低于 100%，确认所收取的保费覆盖了预期损失，同时定价步骤保持简洁且可审计。

## 数据来源

| 数据集 | 行数 | 描述 | 关键变量 |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | 用 `rand()` 内联生成的合成在保财产与意外险保单组合 | `policy_id`、`region`（城市/郊区/农村）、`years_insured`、`exposure`（车年）、`claim_count`（泊松分布）、`incurred_loss`（伽马严重度 x 次数）、`class_pure_prem`（按地区的人工费率） |

DATA 步循环覆盖更大范围的 `policy_id`，但本环境在无许可证模式下运行，因此输出上限为前 **100 条观测**——下面已定价的账本反映的正是这 100 份保单。

# 使用 PROC FCMP 构建可复用的精算定价函数库

精算师在定价、准备金评估和报告工作中反复进行相同的计算：把损失转换为*纯保费*，施加*费用与风险加成*得到收取保费，用*信度*将保单自身的经验与分类费率混合，并将未来现金流*贴现*到现值。在每个 DATA 步中重复键入这些公式，容易引入复制粘贴错误，也让假设变更变得痛苦。

**PROC FCMP**（SAS 函数编译器）让我们把每个公式定义一次，作为命名函数或子程序，将编译好的例程存储到一个函数库中，然后像调用任何内置 SAS 函数一样调用它们。在本笔记本中，我们将：

1. 用 `PROC FCMP` 编译一个小型精算函数库。
2. 用 `CMPLIB=` 系统选项为本次会话注册该函数库。
3. 生成一份合成的财产与意外险保单组合。
4. 通过在单个 DATA 步中调用我们的自定义函数和一个多输出子程序，为每份保单定价。
5. 汇总并解读已定价的保单组合。

## 步骤 1 —— 生成合成保单组合

我们模拟一批在保车险保单（在无许可证模式的上限下，下面只对前 100 份定价）。每份保单属于一个具有自身*纯保费*（每车年预期损失）的费率地区。理赔次数遵循按风险暴露缩放的泊松过程，严重度遵循伽马分布，因此 `incurred_loss` 是一个真实的复合（泊松 x 伽马）损失。`years_insured` 之后将驱动信度权重。通过 `call streaminit` 设置固定种子，使数据可复现。

In [1]:
数据 portfolio;
    调用 streaminit(20260531);
    长度 region $12;
    标签 policy_id       = "保单编号"
          region           = "地区"
          years_insured    = "承保年限"
          exposure         = "风险暴露(车年)"
          claim_count      = "理赔次数"
          incurred_loss    = "已发生损失"
          class_pure_prem  = "分类纯保费";
    循环 policy_id = 10001 到 12000;
        /* 分配费率地区 */
        u = rand('uniform');
        如果 u < 0.40 那么 循环; region = '城市';    base_pp = 820; lambda = 0.18; 结束;
        否则 如果 u < 0.75 那么 循环; region = '郊区'; base_pp = 540; lambda = 0.11; 结束;
        否则 循环; region = '农村';    base_pp = 360; lambda = 0.07; 结束;

        /* 承保年限与风险暴露(车年) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* 复合理赔过程：泊松频率，伽马严重度 */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        循环 c = 1 到 claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        结束;
        incurred_loss = round(incurred_loss, 0.01);

        /* 该保单风险暴露对应的人工分类纯保费 */
        class_pure_prem = round(base_pp * exposure, 0.01);

        输出;
    结束;
    保留 policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
运行;

过程 打印 数据=portfolio(obs=8) noobs 标签;
    标题 '前 8 份模拟保单';
运行;

                                                       前 8 份模拟保单                                                        

        保单编号      地区          承保年限              风险暴露(车年)          理赔次数            已发生损失            分类纯保费
       10001  农村                 5                  1.36             0                0            489.6
       10002  城市                 8                  0.97             1          3432.28            795.4
       10003  城市                 2                  1.53             2          7155.92           1254.6
       10004  郊区                 9                   2.4             0                0             1296
       10005  农村                 5                  0.99             0                0            356.4
       10006  郊区                 1                  0.83             0                0            448.2
       10007  农村                 5                  0.97             0                0            349.2
       10008  农村                 7    


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.45 seconds
  cpu   0.45 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## 步骤 2 —— 编译精算函数库

现在进入本笔记本的核心。`PROC FCMP` 配合 `OUTLIB=work.actfuncs.pricing`，将四个函数和一个子程序编译进 `work.actfuncs` 数据集的 `pricing` 包中：

- **`pure_premium`** —— 每单位风险暴露的已观测损失（频率 x 严重度的组合）。
- **`credibility_z`** —— 有限波动信度因子 `Z = sqrt(n / (n + k))`，其中 `n` 是保单的风险暴露年数，`k` 是一个调节常数。
- **`charged_premium`** —— 对损失成本施加一个比例风险加成和一个固定费用率，得到我们实际收取的保费。
- **`pv_reserve`** —— 未来理赔给付的现值，`FV / (1+r)**t`，用于对案件准备金贴现。
- **`blend_premium`**（一个 SUBROUTINE）—— 使用 `OUTARGS` 一次返回*两个*值：经信度加权的纯保费，以及所使用的信度因子，使 DATA 步能在一次调用中同时获得两者。

每个例程都以 `ENDSUB` 结束，子程序通过 `OUTARGS` 为其可写参数命名。

In [2]:
过程 fcmp outlib=work.actfuncs.pricing;

    /* 观测纯保费：每单位风险暴露的损失成本 */
    function pure_premium(loss, exposure);
        如果 exposure <= 0 那么 RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* 有限波动信度：Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        如果 n_years <= 0 那么 RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* 收取保费 = 损失成本经风险加成和费用调整后的毛保费 */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        如果 expense_ratio >= 1 那么 RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* 未来理赔给付按利率 r 贴现 t 年后的现值 */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* 信度混合：返回加权纯保费及所使用的 Z */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

运行;


               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## 步骤 3 —— 用 CMPLIB= 注册函数库

仅仅编译例程还不够；必须告诉 SAS，当之后的 DATA 步或某个过程引用一个它无法识别为内置项的名称时，去哪里查找这些例程。`CMPLIB=` 系统选项指向保存编译代码的数据集（而不是三级包名）。在这条 `OPTIONS` 语句之后，上面的每个函数和子程序都可以按名称调用。

In [3]:
选项 cmplib=work.actfuncs;


NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## 步骤 4 —— 用自定义函数为保单组合定价

现在的定价 DATA 步几乎不含算术运算——精算意图直接体现在函数名中。对每份保单，我们：

1. 用 `pure_premium` 计算保单自身的观测纯保费。
2. 调用 `blend_premium` 子程序，将其与所在地区的分类费率进行信度加权，通过 `OUTARGS` 同时取回混合损失成本和信度因子。
3. 用 `charged_premium`，以 12% 的风险加成和 25% 的费用率，把混合损失成本毛算为收取保费。
4. 把仍未决的案件准备金估计为保单已发生损失的 35%，并按 4% 的利率贴现三年得到现值，使用 `pv_reserve`。

注意子程序的输出参数（`blended_pp`、`z`）必须在 `CALL` 之前先初始化。准备金现值因保单而异，因为它取决于每份保单实际发生的损失——无理赔的保单准备金为零，因此其 `reserve_pv` 也为零。

In [4]:
数据 rated;
    设置 portfolio;

    标签 policy_id       = "保单编号"
          region           = "地区"
          years_insured    = "承保年限"
          exposure         = "风险暴露(车年)"
          incurred_loss    = "已发生损失"
          own_pp           = "自身纯保费"
          class_pure_prem  = "分类纯保费"
          blended_pp       = "混合纯保费"
          z                = "信度因子z"
          premium          = "保费"
          case_reserve     = "已决未决准备金"
          reserve_pv       = "准备金现值";

    /* 1. 保单自身的损失经验，表示为纯保费 */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. 将保单自身经验与分类费率进行信度加权。
          k = 4 个风险暴露年，用于接近完全信度。 */
    blended_pp = .;
    z = .;
    调用 blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. 把混合损失成本(每车年)换算为收取保费 */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. 未决案件准备金 = 已发生损失的 35%，预计 3 年后结案；
          按 4% 的利率贴现到现值。 */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    保留 policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
运行;

过程 打印 数据=rated(obs=10) noobs 标签;
    标题 '已定价保单组合(前 10 份)';
    变量 policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
运行;

                                                    已定价保单组合(前 10 份)                                                     

        保单编号      地区          承保年限              风险暴露(车年)            自身纯保费            混合纯保费          信度因子z       保费            准备金现值
       10001  农村                 5                  1.36                0            91.67          0.745   186.18                0
       10002  城市                 8                  0.97          3538.43          3039.59          0.816  4402.95          1067.95
       10003  城市                 2                  1.53          4677.07          3046.88          0.577  6961.51          2226.55
       10004  郊区                 9                   2.4                0            90.69          0.832   325.03                0
       10005  农村                 5                  0.99                0            91.67          0.745   135.52                0
       10006  郊区                 1                  0.83                0            2


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## 步骤 5 —— 汇总已定价的保单组合

由于每份保单都通过同一个编译好的函数库定价，我们可以按地区汇总整个保单组合。我们报告平均收取保费、平均信度因子、总已发生损失，以及贴现后的案件准备金总额，然后推导出各细分市场的隐含赔付率，以观察所收取的保费是否覆盖了预期损失。

In [5]:
过程 均值 数据=rated mean sum maxdec=2 nonobs;
    分类 region;
    变量 premium z incurred_loss reserve_pv;
    标签 region="地区" premium="保费" z="信度因子z" incurred_loss="已发生损失" reserve_pv="准备金现值";
    标题 '按费率地区划分的保单组合汇总';
运行;

/* 隐含赔付率 = 已发生损失 / 收取保费，
   以及各地区承担的准备金现值。 */
过程 SQL;
    标题 '各地区的隐含赔付率与准备金现值';
    选择 region,
           count(*)                              AS n_policies,
           sum(incurred_loss)                    AS total_loss     格式=dollar12.2,
           sum(premium)                          AS total_premium  格式=dollar12.2,
           sum(incurred_loss) / sum(premium)     AS loss_ratio     格式=percent8.1,
           sum(reserve_pv)                       AS total_pv_reserve 格式=dollar12.2
    FROM rated
    GROUP 按照 region
    ORDER 按照 region;
QUIT;
标题;

                                                     按费率地区划分的保单组合汇总                                                     

                                                  The MEANS Procedure

                                           Analysis Variable : premium 保费

        地区                 Mean            Sum
        --------------------------------------
        农村               476.61       11438.62
        城市              1987.17       67563.93
        郊区               813.04       34147.74
        --------------------------------------

                                          Analysis Variable : z 信度因子z

        地区                 Mean            Sum
        --------------------------------------
        农村                 0.71          17.14
        城市                 0.70          23.90
        郊区                 0.68          28.36
        --------------------------------------

                                   Analysis Variable : incurred_loss 已发生损失

        地区         


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## 结果解读

定价 DATA 步从未在代码里直接写出任何一个贴现或信度公式——它只是调用 `pure_premium`、`blend_premium`、`charged_premium` 和 `pv_reserve`。这正是 **PROC FCMP** 的价值所在：精算假设都存放在一个可以单元测试、纳入版本控制、并在定价、准备金评估和报告工作中复用的编译函数库里。修改信度常数 `k`、风险加成或费用率，只需在函数库里改一处，而不必在每个程序里逐一排查。

阅读已定价的保单账本和分地区汇总：

- **信度（`z`）** 随 `years_insured` 上升，正如有限波动公式 `Z = sqrt(n / (n + k))` 所决定的那样。在前十份保单中，承保一年的郊区保单（10006）的 `z = 0.447`，承保两年的城市保单（10003）为 `z = 0.577`，而承保九年的郊区保单（10004）达到 `z = 0.832`。经验单薄的保单被拉向所在地区的分类费率，承保年限长的保单则更依赖自身的损失经验。
- **混合效果：** 无理赔的保单（占账本的大多数）`own_pp = 0`，因此 `blend_premium` 返回的 `blended_pp` 接近分类费率乘以 `(1 - z)`——例如保单 10001（农村，`z = 0.745`）落在 `91.67`，对应 `360`/车年的分类费率。两份确实发生理赔的城市保单 10002 和 10003，则把混合损失成本拉高，趋向各自较高的自身损失经验（`3039.59` 和 `3046.88`）。
- **收取保费** 高于混合损失成本，因为 `charged_premium` 施加了 12% 的风险加成，并按 25% 的费用率毛算，相当于对损失成本乘以固定系数 `1.12 / 0.75 = 1.493`。平均收取保费为**农村 476.61**、**郊区 813.04**、**城市 1,987.17**。
- **贴现准备金：** `pv_reserve` 将每份保单的未决案件准备金（已发生损失的 35%）按 4% 的利率贴现三年，现值系数为 `0.889`。无理赔的保单 `reserve_pv = 0`；两个城市理赔保单分别贡献 `1,067.95` 和 `2,226.55`。汇总后，整个保单组合持有的贴现准备金为**农村 $2,151.56**、**郊区 $5,932.52**、**城市 $13,364.13**。
- **隐含赔付率** 落在**农村 60.5%**、**郊区 55.8%**、**城市 63.6%**——都稳妥地低于 100%，说明所收取的保费在每个细分市场都覆盖了预期损失。城市细分市场的赔付率最高，是由其两笔大额模拟损失驱动的；真实的费率复核会先检验这一信号在更多出险年份中是否持续，再考虑调整人工费率。

`blend_premium` 子程序还展示了用 `OUTARGS` 从一次 `CALL` 中返回多个结果的用法——混合损失成本和信度因子 `z` 一起返回——避免了分开调用函数，让每份保单的定价逻辑保持紧凑且可审计。